In [ ]:
import duckdb
import pandas as pd
import numpy as np
import country_converter as coco
import io, requests

con = duckdb.connect("data/etf.duckdb", read_only=True)

try:
    df_obs = con.execute("SELECT * FROM silver.observations WHERE family = 'country'").df()
    cc = coco.CountryConverter()
    target_codes = df_obs[['metric', 'code']].drop_duplicates()
    target_codes["iso3"] = cc.pandas_convert(series=target_codes["code"], to="ISO3")
    target_countries = sorted([c for c in target_codes["iso3"].dropna().unique() if c != "not found"])
except Exception as e:
    print(f"Warning: Could not read silver.observations ({e}). Falling back to global sovereign universe.")
    cc = coco.CountryConverter()
    target_countries = sorted(cc.data['ISO3'].dropna().unique().tolist())

print(f"Initialized {len(target_countries)} target countries for ingestion loops.")

In [ ]:
con.execute("CHECKPOINT;")
con.execute("VACUUM;")
con.close()

In [ ]:
import wbgapi as wb

wb.get_options['timeout'] = 25
wb.get_options['headers'] = {'User-Agent': 'Mozilla/5.0'}

wb_indicators = [
    # --- 1. HEALTH INDEX (Dalio p. 136) & POPULATION NORMALIZERS ---
    "SP.DYN.LE00.IN",      # Life expectancy at birth (years)
    "SH.DYN.MORT",        # Child mortality rate (per 1,000 live births)
    "EN.ATM.PM25.MC.M3",   # Air pollution PM2.5 mean annual exposure (ug/m3)
    "SH.H2O.SMDW.ZS",     # Pop using safely managed drinking water (%)
    "SH.H2O.BASW.ZS",     # Pop using basic drinking water services (%)
    "SH.STA.SMSS.ZS",     # Pop using safely managed sanitation (%)
    "SH.STA.BASS.ZS",     # Pop using basic sanitation facilities (%)
    "SN.ITK.DEFC.ZS",     # Food insecurity / undernourishment (%)
    "VC.IHR.PSRC.P5",     # Homicide rate (per 100,000 people)
    "VC.IDP.TOCV",        # Population displaced by conflict (headcount)
    "SH.UHC.SRVS.CV.XD",   # Universal health coverage (UHC) service coverage index
    "SP.POP.TOTL",         # Total population (required normalizer for headcounts)

    # --- 2. HAPPINESS INDEX (Dalio p. 136 / Data Source #56) ---
    "SH.STA.SUIC.P5",     # Suicide mortality rate (per 100,000 people)

    # --- 3. 10Y GROWTH DEMOGRAPHICS & WORKFORCE (Dalio pp. 148, 166) ---
    "SP.POP.1564.TO.ZG",   # Working-age population growth 15-64 annual % (Delta Workers)
    "SP.POP.1564.TO",      # Population ages 15-64 total
    "SP.POP.1564.ZS",      # Population ages 15-64 (% of total)
    "SP.POP.DPND",         # Age dependency ratio (% of working-age population)
    "SP.POP.DPND.OL",      # Age dependency ratio, old (% of working-age population)
    "SP.POP.GROW",         # Total population growth (annual %)

    # --- 4. ECONOMIC OUTPUT & PRODUCTIVITY (Dalio pp. 132, 161) ---
    "NY.GDP.MKTP.PP.KD",   # GDP, PPP (constant 2017 int$) [Scale]
    "NY.GDP.MKTP.CD",      # GDP, nominal US$ [Scale]
    "NY.GDP.PCAP.PP.KD",   # GDP per capita, PPP (constant 2017 int$) [Quality]
    "NY.GDP.PCAP.CD",      # GDP per capita, nominal US$
    "NY.GDP.MKTP.KD.ZG",   # Real GDP growth rate (annual %)
    "SL.GDP.PCAP.EM.KD",   # Real GDP per person employed (labor productivity)
    "NV.AGR.TOTL.ZS",      # Agriculture, forestry, fishing value added (% GDP)
    "NV.IND.TOTL.ZS",      # Industry value added (% GDP)

    # --- 5. TRADE STRENGTH & OPENNESS (Dalio pp. 132, 134) ---
    "BX.GSR.GNFS.CD",      # Exports of goods & services (current US$)
    "BM.GSR.GNFS.CD",      # Imports of goods & services (current US$)
    "TX.VAL.MRCH.CD.WT",   # Merchandise exports (current US$)
    "TX.MNF.TECH.ZS.UN",   # Medium & high-tech exports (% mfg exports)
    "NE.TRD.GNFS.ZS",      # Trade openness (% GDP)
    "TM.TAX.MRCH.WM.AR.ZS",# Tariff rate, applied, weighted mean all products (%)

    # --- 6. INNOVATION & TECHNOLOGY (Dalio pp. 132, 179) ---
    "IP.PAT.RESD",         # Patent applications, residents
    "IP.PAT.NRES",         # Patent applications, non-residents
    "BX.GSR.ROYL.CD",      # IP royalty & license fee receipts (current US$)
    "BM.GSR.ROYL.CD",      # IP royalty & license fee payments (current US$)
    "GB.XPD.RSDV.GD.ZS",   # Research & development expenditure (% GDP)
    "SP.POP.SCIE.RD.P6",   # Researchers in R&D (per million people)
    "NV.MNF.TECH.ZS.UN",   # High-tech manufacturing value added (% MVA)
    "IP.TMK.RSCT",         # Trademark applications, resident
    "IC.BUS.NREG",         # New businesses registered (commercialism)
    "IT.NET.USER.ZS",      # Internet users (% population)
    "IT.NET.SECR.P6",      # Secure internet servers (per million people)
    "IT.NET.BBND",         # Fixed broadband subscriptions

    # --- 7. MILITARY STRENGTH & CAPABILITIES (Dalio pp. 132, 134) ---
    "MS.MIL.XPND.CD",      # Military expenditure (current US$)
    "MS.MIL.XPND.GD.ZS",   # Military expenditure (% GDP)
    "MS.MIL.XPND.ZS",      # Military expenditure (% central gov expenditure)
    "MS.MIL.TOTL.P1",      # Active armed forces personnel total
    "MS.MIL.MPRT.KD",      # SIPRI conventional arms imports
    "MS.MIL.XPRT.KD",      # SIPRI conventional arms exports

    # --- 8. MARKETS & FINANCIAL CENTER (Dalio p. 132) ---
    "CM.MKT.LCAP.CD",      # Listed stock market capitalization (current US$)
    "CM.MKT.LCAP.GD.ZS",   # Market capitalization (% GDP)
    "CM.MKT.TRAD.CD",      # Stock market total value traded (current US$)
    "FS.AST.PRVT.GD.ZS",   # Domestic private credit (% GDP)
    "FB.BNK.CAPA.ZS",      # Bank capital to assets ratio (%)
    "FB.AST.NPER.ZS",      # Bank nonperforming loans (% total gross loans)
    "BX.PEF.TOTL.CD.WD",   # Portfolio equity net inflows (current US$)
    "FB.CBK.BRCH.P5",      # Commercial bank branches (per 100,000 adults)

    # --- 9. EDUCATION (Dalio pp. 132, 162) ---
    "SE.TER.ENRR",         # School enrollment, tertiary (% gross)
    "SE.SEC.ENRL",         # School enrollment, secondary (total headcount)
    "SE.PRM.ENRL",         # School enrollment, primary (total headcount)
    "SE.XPD.TOTL.GD.ZS",   # Government expenditure on education (% GDP)
    "HD.HCI.EYRS",         # Human Capital Index: Expected years of schooling
    "HD.HCI.OVRL",         # Human Capital Index overall (0-1)
    "SE.COM.DURS",         # Compulsory education duration (years)

    # --- 10. INFRASTRUCTURE & INVESTMENT (Dalio pp. 132, 158) ---
    "NE.GDI.FTOT.CD",      # Gross fixed capital formation (current US$)
    "NE.GDI.FTOT.ZS",      # Gross capital formation / Investment rate (% GDP)
    "NY.GNS.ICTR.ZS",      # Gross national savings rate (% GDP)
    "EG.ELC.ACCS.ZS",      # Access to electricity (% population)
    "EG.ELC.PROD.KH",      # Electricity production (kWh)
    "IS.AIR.PSGR",         # Air transport passengers carried
    "IS.AIR.GOOD.MT.K1",   # Air transport freight (million ton-km)
    "IS.RRS.GOOD.MT.K6",   # Railways goods transported (million ton-km)
    "IS.RRS.TOTL.KM",      # Rail lines total route-km
    "LP.LPI.OVRL.XQ",      # Logistics Performance Index overall (1-5 scale)

    # --- 11. GEOLOGY & ENVIRONMENT (Dalio p. 133) ---
    "AG.LND.TOTL.K2",      # Land area (sq km)
    "ER.H2O.INTR.K3",      # Internal renewable freshwater resources (billion m3)
    "ER.H2O.INTR.PC",      # Renewable freshwater per capita (m3)
    "EN.ATM.CO2E.KT",      # Total CO2 emissions (kt)
    "EN.ATM.CO2E.PC",      # CO2 emissions per capita (metric tons)
    "TX.VAL.MMTL.ZS.UN",   # Ores and metals exports (% merchandise exports)
    "TM.VAL.MMTL.ZS.UN",   # Ores and metals imports (% merchandise imports)

    # --- 12. LABOR, CULTURE, & INEQUALITY (Dalio pp. 133, 143, 181) ---
    "SL.TLF.CACT.ZS",      # Labor force participation rate, total (%)
    "SL.TLF.CACT.MA.ZS",   # Labor force participation rate, male (%)
    "SL.UEM.TOTL.ZS",      # Unemployment rate (% total labor force)
    "SI.POV.GINI",         # Gini index (wealth & income inequality)
    "SI.DST.FRST.20",      # Income share held by lowest 20%
    "SI.DST.05TH.20",      # Income share held by highest 20%
    "SI.DST.10TH.10",      # Income share held by highest 10%

    # --- 13. INDEBTEDNESS & MONETARY SOLVENCY (Dalio pp. 133, 141) ---
    "FS.AST.CGOV.GD.ZS",   # Domestic claims on central government (% GDP)
    "BN.CAB.XOKA.GD.ZS",   # Current account balance (% GDP)
    "FI.RES.TOTL.MO",      # Total foreign reserves in months of imports
    "FP.CPI.TOTL.ZG",      # Inflation, consumer prices (annual %)
    "NE.CON.GOVT.ZS",      # General government consumption expenditure (% GDP)

    # --- 14. WORLDWIDE GOVERNANCE INDICATORS (WGI / Dalio p. 133) ---
    "RL.EST",              # Rule of Law estimate (-2.5 to +2.5)
    "CC.EST",              # Control of Corruption estimate (-2.5 to +2.5)
    "RQ.EST",              # Regulatory Quality / Bureaucracy estimate
    "GE.EST",              # Government Effectiveness estimate
    "PV.EST",              # Political Stability & Absence of Violence
    "VA.EST"               # Voice & Accountability
]

wb_indicators = sorted(list(set(wb_indicators)))
wb_results = []
print(f"Starting World Bank fetch for {len(wb_indicators)} indicators across {len(target_countries)} countries...")

for i, iso3 in enumerate(target_countries, 1):
    try:
        # Pull across 2018-2026 to capture multi-year structural releases
        df_c = wb.data.DataFrame(wb_indicators, economy=iso3, time=range(2018, 2026), numericTimeKeys=True)
        if not df_c.empty:
            df_c = df_c.reset_index()
            # Unpivot wide year columns to standard tall schema
            year_cols = [c for c in df_c.columns if isinstance(c, int) or (isinstance(c, str) and c.isdigit())]
            df_melted = pd.melt(
                df_c,
                id_vars=["series"],
                value_vars=year_cols,
                var_name="year",
                value_name="value"
            ).dropna(subset=["value"])
            
            df_melted["country_code"] = iso3
            df_melted["source"] = "WB"
            df_melted.rename(columns={"series": "indicator_code"}, inplace=True)
            wb_results.append(df_melted[["country_code", "indicator_code", "year", "value", "source"]])

        if i % 15 == 0 or i == len(target_countries):
            print(f"  [{i}/{len(target_countries)}] Processed {iso3}")
    except Exception as e:
        print(f"  [{i}/{len(target_countries)}] {iso3} failed: {e}")
        continue

df_wb_final = pd.concat(wb_results, ignore_index=True) if wb_results else pd.DataFrame()
print(f"World Bank: Ingested {len(df_wb_final)} observations across {df_wb_final['country_code'].nunique()} countries.")

In [ ]:
imf_results = []
imf_indicators = "GGXWDG_NGDP+GGXCNL_NGDP+NGDP_RPCH+BCA_NGDPD+PCPIPCH+LUR"

print(f"Starting IMF SDMX 3.0 loop across {len(target_countries)} countries...")

for i, iso3 in enumerate(target_countries, 1):
    url = f"https://api.imf.org/external/sdmx/3.0/data/dataflow/IMF.RES/WEO/~/{iso3}.{imf_indicators}.A"
    params = {"startPeriod": "2018", "endPeriod": "2026"}
    headers = {"Accept": "text/csv"}

    try:
        resp = requests.get(url, params=params, headers=headers, timeout=15)
        if resp.status_code == 200 and len(resp.text.strip()) > 0:
            df_c = pd.read_csv(io.StringIO(resp.text))
            geo_col = "COUNTRY" if "COUNTRY" in df_c.columns else "REF_AREA"
            
            if not df_c.empty and "OBS_VALUE" in df_c.columns and geo_col in df_c.columns:
                df_sub = df_c[[geo_col, "INDICATOR", "TIME_PERIOD", "OBS_VALUE"]].copy()
                df_sub.columns = ["country_code", "indicator_code", "year", "value"]
                df_sub["indicator_code"] = "IMF." + df_sub["indicator_code"].astype(str)
                df_sub["value"] = pd.to_numeric(df_sub["value"], errors="coerce")
                df_sub["year"] = pd.to_numeric(df_sub["year"], errors="coerce")
                df_sub["source"] = "IMF"
                imf_results.append(df_sub.dropna())
                
        if i % 15 == 0 or i == len(target_countries):
            print(f"  [{i}/{len(target_countries)}] Processed {iso3}")
            
    except Exception as e:
        print(f"  [{i}/{len(target_countries)}] Failed {iso3}: {e}")
        continue

df_imf_final = pd.concat(imf_results, ignore_index=True) if imf_results else pd.DataFrame()
print(f"IMF WEO: Ingested {len(df_imf_final)} observations across {df_imf_final['country_code'].nunique()} countries.")

In [ ]:
API_KEY = "kJpbTkcdzNctTcTMKHJF9bWdkwbwmqHWJFfRnKGL"
eia_rows = []

print(f"Starting US EIA API v2 loop across {len(target_countries)} countries...")

for i, iso3 in enumerate(target_countries, 1):
    url = "https://api.eia.gov/v2/international/data"
    params = {
        "api_key": API_KEY,
        "frequency": "annual",
        "data[0]": "value",
        "facets[activityId][]": ["1", "2"],
        "facets[productId][]": "44",
        "facets[countryRegionId][]": iso3,
        "facets[unit][]": "QBTU",
        "start": "2018",
        "end": "2026"
    }

    try:
        res = requests.get(url, params=params, timeout=15).json()
        data = res.get("response", {}).get("data", [])
        if data:
            df_c = pd.DataFrame(data)[["period", "countryRegionId", "activityId", "value"]]
            eia_rows.append(df_c)
        if i % 15 == 0 or i == len(target_countries):
            print(f"  [{i}/{len(target_countries)}] Processed {iso3}")
    except Exception:
        continue

eia_results = []
if eia_rows:
    df_eia_raw = pd.concat(eia_rows, ignore_index=True)
    df_pivot = df_eia_raw.pivot(index=["countryRegionId", "period"], columns="activityId", values="value").reset_index()
    
    # Rename activity columns
    df_pivot.rename(columns={"countryRegionId": "country_code", "period": "year", "1": "prod_qbtu", "2": "cons_qbtu"}, inplace=True)
    df_pivot["prod_qbtu"] = pd.to_numeric(df_pivot.get("prod_qbtu", np.nan), errors="coerce")
    df_pivot["cons_qbtu"] = pd.to_numeric(df_pivot.get("cons_qbtu", np.nan), errors="coerce")
    df_pivot["year"] = pd.to_numeric(df_pivot["year"], errors="coerce")
    
    # Ingest Total Primary Energy Production (Scale)
    df_prod = df_pivot[["country_code", "year", "prod_qbtu"]].dropna().copy()
    df_prod.rename(columns={"prod_qbtu": "value"}, inplace=True)
    df_prod["indicator_code"] = "EIA.ENERGY.PRODUCTION_QBTU"
    df_prod["source"] = "EIA"
    eia_results.append(df_prod)

    # Ingest Net Energy Self-Sufficiency Ratio (Dalio p. 133 / p. 135)
    df_suff = df_pivot.dropna(subset=["prod_qbtu", "cons_qbtu"]).copy()
    df_suff = df_suff[df_suff["cons_qbtu"] > 0].copy()
    df_suff["value"] = (df_suff["prod_qbtu"] - df_suff["cons_qbtu"]) / df_suff["cons_qbtu"]
    df_suff["indicator_code"] = "EIA.ENERGY.SELF_SUFF"
    df_suff["source"] = "EIA"
    eia_results.append(df_suff[["country_code", "indicator_code", "year", "value", "source"]])

df_eia_final = pd.concat(eia_results, ignore_index=True) if eia_results else pd.DataFrame()
print(f"EIA: Ingested {len(df_eia_final)} observations across {df_eia_final['country_code'].nunique()} countries.")

In [ ]:
import io
import requests
import zipfile
import pandas as pd

url_vdem = "https://www.v-dem.net/media/datasets/V-Dem-CY-FullOthers-v16_csv.zip"
print("Streaming V-Dem v16 Full+Others archive into memory...")
res = requests.get(url_vdem, headers={"User-Agent": "Mozilla/5.0"}, timeout=180)
res.raise_for_status()

vdem_indicators = [
    "v2cltrnslw", "v2excrptps", "v2exthftps", 
    "v2clrspct", "v2x_rule", "v2cacamps", "v2caviol"
]
use_cols = ["country_text_id", "year"] + vdem_indicators

with zipfile.ZipFile(io.BytesIO(res.content)) as z:
    with z.open("V-Dem-CY-Full+Others-v16.csv") as f:
        df_vdem_master = pd.read_csv(f, usecols=use_cols, low_memory=False)

df_vdem_sub = df_vdem_master[
    (df_vdem_master["country_text_id"].isin(target_countries)) & 
    (df_vdem_master["year"] >= 2018)
].copy()

df_vdem_final = pd.melt(
    df_vdem_sub,
    id_vars=["country_text_id", "year"],
    value_vars=vdem_indicators,
    var_name="indicator_code",
    value_name="value"
).dropna()

df_vdem_final.rename(columns={"country_text_id": "country_code"}, inplace=True)
df_vdem_final["indicator_code"] = "VDEM." + df_vdem_final["indicator_code"]
df_vdem_final["source"] = "VDEM"

print(f"V-Dem: Extracted 7 governance & conflict indicators ({len(df_vdem_final)} rows) across {df_vdem_final['country_code'].nunique()} countries.")

In [ ]:
url_gain = "https://gain.nd.edu/assets/647440/ndgain_countryindex_2026.zip"
print("Streaming ND-GAIN 2026 archive into memory...")
res = requests.get(url_gain, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)

ndgain_frames = []
metrics = ["gain", "readiness", "vulnerability"]

with zipfile.ZipFile(io.BytesIO(res.content)) as z:
    names = z.namelist()
    for m in metrics:
        file_path = next((name for name in names if name.endswith(f"{m}.csv")), None)
        if not file_path:
            continue
        df_m = pd.read_csv(z.open(file_path))

        df_c = df_m[df_m["ISO3"].isin(target_countries)].copy()
        if not df_c.empty:
            year_cols = [c for c in df_c.columns if c.isdigit() and int(c) >= 2018]
            df_melted = pd.melt(df_c, id_vars=["ISO3"], value_vars=year_cols, var_name="year", value_name="value").dropna()
            df_melted["indicator_code"] = f"NDGAIN.{m.upper()}"
            df_melted["source"] = "NDGAIN"
            df_melted.rename(columns={"ISO3": "country_code"}, inplace=True)
            ndgain_frames.append(df_melted[["country_code", "indicator_code", "year", "value", "source"]])

df_ndgain_final = pd.concat(ndgain_frames, ignore_index=True) if ndgain_frames else pd.DataFrame()
print(f"ND-GAIN: Extracted 3 climate indicators ({len(df_ndgain_final)} rows) across {df_ndgain_final['country_code'].nunique()} countries.")

In [ ]:
if 'iso3' not in df_obs.columns:
    iso2_to_iso3 = target_codes.set_index("code")["iso3"].to_dict()
    df_obs['iso3'] = df_obs['code'].map(iso2_to_iso3)

obs_counts = df_obs['iso3'].value_counts()
country_weights = pd.Series(0, index=target_countries, dtype=float)
for c, cnt in obs_counts.items():
    if c in country_weights.index:
        country_weights[c] = cnt

total_obs_weight = country_weights.sum()
country_weight_shares = country_weights / total_obs_weight

all_dfs = [df_wb_final, df_imf_final, df_eia_final, df_vdem_final, df_ndgain_final]
df_master = pd.concat([d for d in all_dfs if not d.empty], ignore_index=True)
df_master["year"] = df_master["year"].astype(int)
df_master["value"] = pd.to_numeric(df_master["value"], errors="coerce")
df_master.dropna(subset=["value"], inplace=True)

total_target_count = len(target_countries)
recent_year = 2024

gauge_mapping = {
    # Health Index (Gauge 20)
    "SP.DYN.LE00.IN": "20. Health Index", "SH.DYN.MORT": "20. Health Index",
    "EN.ATM.PM25.MC.M3": "20. Health Index", "SH.H2O.SMDW.ZS": "20. Health Index",
    "SH.H2O.BASW.ZS": "20. Health Index", "SH.STA.SMSS.ZS": "20. Health Index",
    "SH.STA.BASS.ZS": "20. Health Index", "SN.ITK.DEFC.ZS": "20. Health Index",
    "VC.IHR.PSRC.P5": "20. Health Index", "VC.IDP.TOCV": "20. Health Index",
    "SH.UHC.SRVS.CV.XD": "20. Health Index",
    # Happiness Index (Gauge 21)
    "SH.STA.SUIC.P5": "21. Happiness Index (Suicide component)",
    # Economic Output (Gauge 5)
    "NY.GDP.MKTP.PP.KD": "05. Economic Output", "NY.GDP.MKTP.CD": "05. Economic Output",
    "NY.GDP.PCAP.PP.KD": "05. Economic Output", "NY.GDP.PCAP.CD": "05. Economic Output",
    "NY.GDP.MKTP.KD.ZG": "05. Economic Output", "NV.AGR.TOTL.ZS": "05. Economic Output",
    "NV.IND.TOTL.ZS": "05. Economic Output", "IMF.NGDP_RPCH": "05. Economic Output",
    # Trade (Gauge 7)
    "BX.GSR.GNFS.CD": "07. Trade", "BM.GSR.GNFS.CD": "07. Trade",
    "TX.VAL.MRCH.CD.WT": "07. Trade", "TX.MNF.TECH.ZS.UN": "07. Trade",
    "NE.TRD.GNFS.ZS": "07. Trade", "TM.TAX.MRCH.WM.AR.ZS": "07. Trade",
    # Innovation & Technology (Gauge 2)
    "IP.PAT.RESD": "02. Innovation & Technology", "IP.PAT.NRES": "02. Innovation & Technology",
    "BX.GSR.ROYL.CD": "02. Innovation & Technology", "BM.GSR.ROYL.CD": "02. Innovation & Technology",
    "GB.XPD.RSDV.GD.ZS": "02. Innovation & Technology", "SP.POP.SCIE.RD.P6": "02. Innovation & Technology",
    "NV.MNF.TECH.ZS.UN": "02. Innovation & Technology", "IP.TMK.RSCT": "02. Innovation & Technology",
    "IC.BUS.NREG": "02. Innovation & Technology", "IT.NET.USER.ZS": "02. Innovation & Technology",
    "IT.NET.SECR.P6": "02. Innovation & Technology", "IT.NET.BBND": "02. Innovation & Technology",
    # Military Strength (Gauge 8)
    "MS.MIL.XPND.CD": "08. Military Strength", "MS.MIL.XPND.GD.ZS": "08. Military Strength",
    "MS.MIL.XPND.ZS": "08. Military Strength", "MS.MIL.TOTL.P1": "08. Military Strength",
    "MS.MIL.MPRT.KD": "08. Military Strength", "MS.MIL.XPRT.KD": "08. Military Strength",
    "FAS.NUCLEAR_STOCKPILE": "08. Military Strength",
    # Markets & Financial Center (Gauge 9)
    "CM.MKT.LCAP.CD": "09. Financial Center", "CM.MKT.LCAP.GD.ZS": "09. Financial Center",
    "CM.MKT.TRAD.CD": "09. Financial Center", "FS.AST.PRVT.GD.ZS": "09. Financial Center",
    "FB.BNK.CAPA.ZS": "09. Financial Center", "FB.AST.NPER.ZS": "09. Financial Center",
    "BX.PEF.TOTL.CD.WD": "09. Financial Center", "FB.CBK.BRCH.P5": "09. Financial Center",
    # Reserve Currency Status (Gauge 10)
    "IMF.COFER_SHARE": "10. Reserve Currency Status", "BIS.FX_TURNOVER_SHARE": "10. Reserve Currency Status",
    # Education (Gauge 1)
    "SE.TER.ENRR": "01. Education", "SE.SEC.ENRL": "01. Education",
    "SE.PRM.ENRL": "01. Education", "SE.XPD.TOTL.GD.ZS": "01. Education",
    "HD.HCI.EYRS": "01. Education", "HD.HCI.OVRL": "01. Education", "SE.COM.DURS": "01. Education",
    # Infrastructure & Investment (Gauge 4)
    "NE.GDI.FTOT.CD": "04. Infrastructure & Investment", "NE.GDI.FTOT.ZS": "04. Infrastructure & Investment",
    "NY.GNS.ICTR.ZS": "04. Infrastructure & Investment", "EG.ELC.ACCS.ZS": "04. Infrastructure & Investment",
    "EG.ELC.PROD.KH": "04. Infrastructure & Investment", "IS.AIR.PSGR": "04. Infrastructure & Investment",
    "IS.AIR.GOOD.MT.K1": "04. Infrastructure & Investment", "IS.RRS.GOOD.MT.K6": "04. Infrastructure & Investment",
    "IS.RRS.TOTL.KM": "04. Infrastructure & Investment", "LP.LPI.OVRL.XQ": "04. Infrastructure & Investment",
    # Geology & Resources (Gauge 15)
    "AG.LND.TOTL.K2": "15. Geology & Resources", "ER.H2O.INTR.K3": "15. Geology & Resources",
    "ER.H2O.INTR.PC": "15. Geology & Resources", "EN.ATM.CO2E.KT": "15. Geology & Resources",
    "EN.ATM.CO2E.PC": "15. Geology & Resources", "TX.VAL.MMTL.ZS.UN": "15. Geology & Resources",
    "TM.VAL.MMTL.ZS.UN": "15. Geology & Resources", "EIA.ENERGY.PRODUCTION_QBTU": "15. Geology & Resources",
    # Cost Competitiveness & Energy (Gauge 3)
    "EIA.ENERGY.SELF_SUFF": "03. Cost Competitiveness & Energy",
    # Indebtedness & Debt Cycles (Gauge 11)
    "IMF.GGXWDG_NGDP": "11. Indebtedness & Debt Cycles", "IMF.GGXCNL_NGDP": "11. Indebtedness & Debt Cycles",
    "FS.AST.CGOV.GD.ZS": "11. Indebtedness & Debt Cycles", "BN.CAB.XOKA.GD.ZS": "11. Indebtedness & Debt Cycles",
    "IMF.BCA_NGDPD": "11. Indebtedness & Debt Cycles", "FI.RES.TOTL.MO": "11. Indebtedness & Debt Cycles",
    "FP.CPI.TOTL.ZG": "11. Indebtedness & Debt Cycles", "IMF.PCPIPCH": "11. Indebtedness & Debt Cycles",
    "NE.CON.GOVT.ZS": "11. Indebtedness & Debt Cycles",
    # Internal Conflict (Gauge 12)
    "VDEM.v2cacamps": "12. Internal Conflict (Polarization)", "VDEM.v2caviol": "12. Internal Conflict (Violence)",
    "PV.EST": "12. Internal Conflict (Stability)",
    # Gaps in Wealth & Values (Gauge 14)
    "SI.POV.GINI": "14. Gaps in Wealth & Values", "SI.DST.FRST.20": "14. Gaps in Wealth & Values",
    "SI.DST.05TH.20": "14. Gaps in Wealth & Values", "SI.DST.10TH.10": "14. Gaps in Wealth & Values",
    # Rule of Law & Governance (Gauge 13)
    "RL.EST": "13. Rule of Law & Governance", "CC.EST": "13. Rule of Law & Governance",
    "RQ.EST": "13. Rule of Law & Governance", "GE.EST": "13. Rule of Law & Governance",
    "VA.EST": "13. Rule of Law & Governance", "VDEM.v2cltrnslw": "13. Rule of Law & Governance",
    "VDEM.v2excrptps": "13. Rule of Law & Governance", "VDEM.v2exthftps": "13. Rule of Law & Governance",
    "VDEM.v2clrspct": "13. Rule of Law & Governance", "VDEM.v2x_rule": "13. Rule of Law & Governance",
    # Character, Civility & Self Sufficiency (Gauge 16)
    "SL.TLF.CACT.ZS": "16. Character, Civility & Self Sufficiency",
    "SL.TLF.CACT.MA.ZS": "16. Character, Civility & Self Sufficiency",
    # Resource Allocation Efficiency (Gauge 17)
    "SL.UEM.TOTL.ZS": "17. Resource Allocation Efficiency", "IMF.LUR": "17. Resource Allocation Efficiency",
    # Productivity / Demographics (Gauge 6)
    "SP.POP.1564.TO.ZG": "06. Productivity Growth (Demographics)",
    "SL.GDP.PCAP.EM.KD": "06. Productivity Growth (Output per worker)",
    # Acts of Nature / Climate (Gauge 18)
    "NDGAIN.GAIN": "18. Acts of Nature & Climate", "NDGAIN.READINESS": "18. Acts of Nature & Climate",
    "NDGAIN.VULNERABILITY": "18. Acts of Nature & Climate"
}

records = []
for ind, grp in df_master.groupby("indicator_code"):
    rep_countries = set(grp["country_code"].unique())
    unique_count = len(rep_countries.intersection(target_countries))
    pct_coverage = (unique_count / total_target_count) * 100
    
    weighted_coverage = country_weight_shares[country_weight_shares.index.isin(rep_countries)].sum() * 100
    
    recent_grp = grp[grp["year"] >= recent_year]
    recent_countries = set(recent_grp["country_code"].unique())
    unique_recent_count = len(recent_countries.intersection(target_countries))
    pct_recent_cov = (unique_recent_count / total_target_count) * 100
    
    weighted_recent_cov = country_weight_shares[country_weight_shares.index.isin(recent_countries)].sum() * 100
    
    latest_yr = grp["year"].max()
    std_val = grp["value"].std()
    distinct_vals = grp["value"].nunique()
    
    records.append({
        "gauge": gauge_mapping.get(ind, "Other / Diagnostic"),
        "indicator_code": ind,
        "source": grp["source"].iloc[0],
        "reporting_countries": unique_count,
        "country_cov_%": round(pct_coverage, 1),
        "weighted_cov_%": round(weighted_coverage, 1),
        f"recent_cov_% (>={recent_year})": round(pct_recent_cov, 1),
        f"weighted_recent_% (>={recent_year})": round(weighted_recent_cov, 1),
        "latest_year": latest_yr,
        "degenerate": (std_val <= 0) and (distinct_vals < 5)
    })

df_health = pd.DataFrame(records).sort_values(
    by=["gauge", "weighted_cov_%", f"weighted_recent_% (>={recent_year})"], 
    ascending=[True, False, False]
)
df_health

In [ ]:
temp = df_health[~(df_health['latest_year'] < recent_year) & ~(df_health['latest_year'] > 2026)]
x, c, v, cov, covr = [], [], [], [], []
for z in list(range(1,301,1)):
    z = z/100
    for col in ['weighted_cov_%', f'weighted_recent_% (>={recent_year})']:
        iqr = temp[col].quantile(0.75) - temp[col].quantile(0.25)
        z=1.5
        temp[f'fail_{col}'] = (temp[col] < (temp[col].median() - iqr*z))
    count = len(temp[temp.select_dtypes(include="bool").sum(axis=1) == 0])
    variety = len(temp[temp.select_dtypes(include="bool").sum(axis=1) == 0]['gauge'].unique())
    cov1 = temp[temp.select_dtypes(include="bool").sum(axis=1) == 0]['weighted_cov_%'].min()
    covr1 = temp[temp.select_dtypes(include="bool").sum(axis=1) == 0][f'weighted_recent_% (>={recent_year})'].min()
    x.append(z)
    c.append(count)
    v.append(variety)
    cov.append(cov1)
    covr.append(covr1)

# z = .72

In [ ]:
testing = pd.DataFrame([x,c,v,cov,covr]).T
testing.columns = ['z_score', 'indicator_count', 'gauge_count', 'min_cov', 'min_recent_cov']
testing

In [ ]:
from matplotlib import pyplot

pyplot.plot(x, y)
pyplot.show()
pyplot.plot(x, v)
pyplot.show()